# Question - Gaussian Mixture Clustering as Conditional Updating
## 1. Deriving the Marginal Density

By the law of total probability, the marginal probability density $p(x_i)$ of an observed data point $X_i$ is found by summing the joint probability density $p(x_i, C_i = k)$ over all possible latent cluster assignments $k \in \{1, \dots, K\}$:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k)$$

Using the definition of conditional probability, we can factor the joint distribution $p(x_i, C_i = k)$ into the product of the prior probability of the cluster and its conditional likelihood:

$$p(x_i, C_i = k) = P(C_i = k) \cdot p(x_i \mid C_i = k)$$

Using the specified parameters of our model:
* The prior probability of cluster membership is $P(C_i = k) = \phi_k$.
* The conditional likelihood of the observation is multivariate Gaussian: $p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$.

Substituting these definitions back into the summation yields the marginal density:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$


### Why this density is called a "Gaussian mixture density"
This formulation is called a **Gaussian mixture density** because it mathematically "mixes" multiple distinct multivariate Gaussian density functions into a single, unified probability density function:
* **The Components:** Each individual term in the sum, $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$, represents a single Gaussian subpopulation (or cluster) with its own mean $\mu_k$ and covariance matrix $\Sigma_k$.
* **The Weights:** The coefficients $\phi_k$ act as mixing weights (or proportions). Since they satisfy the constraints $\phi_k \ge 0$ and $\sum_{k=1}^K \phi_k = 1$, they dictate exactly what fraction of the overall population is contributed by each individual Gaussian component.

---
## 2. Deriving the Posterior Cluster Probability

### Step-by-Step Derivation using Bayes' Rule
Bayes' rule allows us to calculate the conditional probability of a latent cause given an observed event. For a fixed observation $x_i$, the conditional probability that it belongs to a specific cluster $k$ is given by:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{P(X_i = x_i)}$$

By the law of total probability (derived in the previous step), the marginal density in the denominator can be expressed as a sum over all possible clusters $j \in \{1, \dots, K\}$:

$$p(x_i) = \sum_{j=1}^K P(X_i = x_i \mid C_i = j) P(C_i = j)$$

Substituting this summation into the denominator yields the general form of Bayes' rule for this latent variable setup:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^K P(X_i = x_i \mid C_i = j) P(C_i = j)}$$

Next, we substitute our specific model parameters and distributions into the equation:
* The prior cluster probabilities are $P(C_i = k) = \phi_k$ and $P(C_i = j) = \phi_j$.
* The conditional data distributions are multivariate Gaussians: $p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ and $p(x_i \mid C_i = j) = \mathcal{N}(x_i \mid \mu_j, \Sigma_j)$.

Substituting these components directly into the expression gives the final closed-form equation for the responsibility:

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$


### Why $\gamma_{ik}$ is Interpreted as a Posterior Probability of Cluster Membership
The quantity $\gamma_{ik}$ represents a **posterior probability** because of when and how it is calculated relative to our knowledge:
* **Prior vs. Posterior:** Before looking at the data point $x_i$, our prior belief that the point belongs to cluster $k$ is simply $\phi_k$. Once we observe the realization $X_i = x_i$, we update this belief using Bayes' rule. $\gamma_{ik}$ is the updated, conditional probability *after* (posterior to) incorporating the evidence provided by the features of $x_i$.
* **Membership Fraction:** It cleanly acts as a probability distribution over the latent groups for a given data point because it satisfies standard probability axioms: each value is non-negative ($\gamma_{ik} \ge 0$), and summing the responsibilities across all possible clusters for a single data point evaluates exactly to 1 ($\sum_{k=1}^K \gamma_{ik} = 1$).

---
## 3. One-Hot Encoding of the Latent Cluster Variable

### Proof that $\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$
By definition, the conditional expectation of a discrete binary random variable $Z_{ik} \in \{0, 1\}$ given an observation $X_i = x_i$ is computed as the sum of its possible values multiplied by their respective conditional probabilities:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \sum_{z \in \{0,1\}} z \cdot P(Z_{ik} = z \mid X_i = x_i)$$

Expanding this sum for the two possible states ($z=0$ and $z=1$):

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \left(0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)\right) + \left(1 \cdot P(Z_{ik} = 1 \mid X_i = x_i)\right)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(Z_{ik} = 1 \mid X_i = x_i)$$

By definition, the indicator component $Z_{ik} = 1$ if and only if the latent cluster variable $C_i = k$. Therefore, the event $Z_{ik} = 1$ is equivalent to the event $C_i = k$, giving us:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$$



### Vector Derivation for $\mathbb{E}[Z_i \mid X_i = x_i]$
The expectation operator applies element-wise to random vectors. By evaluating the conditional expectation for each component of the latent vector $Z_i$, we stack the individual results:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} P(C_i = 1 \mid X_i = x_i) \\ P(C_i = 2 \mid X_i = x_i) \\ \vdots \\ P(C_i = K \mid X_i = x_i) \end{bmatrix}$$

Recalling from the definition of responsibilities that $\gamma_{ik} = P(C_i = k \mid X_i = x_i)$, we can substitute $\gamma_{ik}$ into each row:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$



### Conclusion
Consequently, we conclude that the soft cluster assignment vector in a Gaussian mixture model is precisely the conditional expectation $\mathbb{E}[Z_i \mid X_i = x_i]$. Instead of selecting a single hard cluster, the conditional expectation yields the posterior probability distribution (responsibilities) across all $K$ groups given the data point $x_i$.

---
## 4. From Soft Assignment to Hard Clustering

### Explanation of the Difference Between Soft Clustering and Hard Clustering

In the context of this model, the fundamental difference between soft and hard clustering lies in how uncertainty regarding cluster membership is handled and represented:

* **Soft Clustering ($\mathbb{E}[Z_i \mid X_i = x_i]$):**
  * Soft clustering retains the full probabilistic information regarding an observation's assignment.
  * Instead of locking a data point into a single group, it expresses cluster membership as a probability distribution or vector of responsibilities $[\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$.
  * Each element $\gamma_{ik}$ satisfies $0 \le \gamma_{ik} \le 1$ and $\sum_{k=1}^K \gamma_{ik} = 1$, representing the partial or fractional membership of the point across all $K$ clusters simultaneously. This is highly useful for points lying on the borders between overlapping clusters.

* **Hard Clustering ($\hat{C}_i$):**
  * Hard clustering forces a discrete, deterministic choice by completely stripping away any classification uncertainty.
  * It maps each data point to one, and only one, specific cluster.
  * Mathematically, this is done by applying the maximum a posteriori (MAP) decision rule: $\hat{C}_i = \arg\max_{1 \le k \le K} \gamma_{ik}$. The algorithm selects the single index $k$ associated with the highest posterior probability and discards the information about the remaining clusters.

---
## 5. Conditional Expectation of the Observation Given the Cluster

### Proof that $\mathbb{E}[X_i \mid C_i = k] = \mu_k$
By the definition of the model, conditional on the latent cluster assignment $C_i = k$, the observation $X_i$ follows a multivariate Gaussian distribution:

$$X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$$

The conditional expectation $\mathbb{E}[X_i \mid C_i = k]$ is by definition the mean parameter of this conditional density function:

$$\mathbb{E}[X_i \mid C_i = k] = \int x_i \cdot \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$


### Why $\mu_k$ Can Be Interpreted as the Center of Cluster $k$
The vector $\mu_k$ represents the expected value or first moment of the Gaussian subpopulation belonging to cluster $k$. Geometrically, because the multivariate normal density is symmetric and unimodal around its mean parameter, $\mu_k$ sits exactly at the peak of the probability distribution for that component. This makes $\mu_k$ the physical center of mass, centroid, or prototypical representative location of cluster $k$ in the $d$-dimensional space $\mathbb{R}^d$.


### Comparison of the Two Conditional Expectations

The two expressions represent opposite directional perspectives of mapping between the observed feature space and the latent cluster space:

1. **$\mathbb{E}[Z_i \mid X_i = x_i]$ (Soft Cluster Membership):**
   * **Direction:** Observed feature space $\rightarrow$ Latent cluster space.
   * **Reasoning:** This expectation conditions on a fixed, known observation $x_i$ and evaluates the state of the one-hot encoded vector $Z_i$. As shown in question 3, its elements are the posterior cluster responsibilities $\gamma_{ik} = P(C_i = k \mid X_i = x_i)$. It answers the inference question: *"Given this specific data point, what is the probability that it originated from each of the clusters?"* Hence, it yields the soft cluster membership of that point.

2. **$\mathbb{E}[X_i \mid C_i = k]$ (Mean Location of a Cluster):**
   * **Direction:** Latent cluster space $\rightarrow$ Observed feature space.
   * **Reasoning:** This expectation conditions on a fixed, known latent cluster identity $C_i = k$ and evaluates the expected position of the feature vector $X_i$. It answers the generative question: *"If we look strictly at the subpopulation of points generated by cluster $k$, what is their average position in the feature space?"* As proven above, this evaluates directly to the parameter $\mu_k$, which defines the mean location of the cluster.

---
## 6. The Complete-Data Likelihood

### Step-by-Step Derivation of the Complete-Data Log-Likelihood
If the latent labels $z_i$ are known, the joint complete-data likelihood for all observations $i = 1, \dots, n$ is given by the product form:

$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

To find the complete-data log-likelihood $\ell_c$, we apply the natural logarithm ($\log$) to both sides of the equation:

$$\ell_c = \log \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

Using the logarithmic identity $\log(\prod a_m) = \sum \log(a_m)$, we convert the outer and inner products into summations:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

Next, we apply the power rule of logarithms ($\log(a^b) = b \log(a)$) to pull the exponent $z_{ik}$ out to the front:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \log \left( \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right)$$

Finally, we use the product rule of logarithms ($\log(ab) = \log a + \log b$) to split the terms inside the parentheses:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$



### Why This Expression is Easy to Maximize if the Values of $z_{ik}$ Were Known

If the indicator variables $z_{ik}$ are known constants rather than hidden variables, optimizing the objective function becomes straightforward due to **decoupling and simplification**:

* **Elimination of the Sum-Inside-Log Problem:** In the standard marginal log-likelihood $\sum \log( \sum \phi_k \mathcal{N} )$, the sum sits *inside* the logarithm. This ties all clusters together and prevents analytical maximization. In the complete-data log-likelihood $\ell_c$, the logarithm applies directly to individual components, converting complex exponential forms into linear or quadratic expressions.
* **Separability of Parameters:** The parameters for each cluster ($\mu_k$, $\Sigma_k$) appear inside independent terms weighted by the constant $z_{ik}$. This means we can maximize the parameters for each cluster $k$ completely independently of the other clusters.
* **Closed-Form Solutions:** With $z_{ik}$ fixed, maximizing $\ell_c$ with respect to $\mu_k$ and $\Sigma_k$ reduces to independent, standard Maximum Likelihood Estimation (MLE) for single Gaussians. The optimal parameters are simply the sample mean and sample covariance of the points explicitly assigned to cluster $k$ (where $z_{ik}=1$).

---
## 7. The EM Interpretation

### Mathematical Setup for the Expected Complete-Data Log-Likelihood
Because the latent labels $z_{ik}$ are unobserved, the Expectation-Maximization (EM) algorithm cannot optimize the complete-data log-likelihood directly. Instead, the E-step resolves this by taking the conditional expectation of the complete-data log-likelihood with respect to the posterior distribution of the latent variables given the current parameter estimates.

Substituting the unknown indicators $z_{ik}$ with their conditional expectations $\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$ yields the auxiliary function $Q$:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$



### Why the E-Step Can Be Interpreted as a Conditional Update of Cluster Membership Probabilities

The Expectation step (E-step) fundamentally functions as a conditional update of the cluster membership probabilities for the following reasons:

* **Dynamic Updating via Bayes' Rule:** The E-step evaluates the responsibilities $\gamma_{ik} = P(C_i = k \mid X_i = x_i)$. This calculation acts as a directional inference step that takes our prior beliefs ($\phi_k$) and modifies them by conditioning on the newly observed physical location of the data point $x_i$, under the framework of the current cluster definitions ($\mu_k, \Sigma_k$).
* **Iterative Refinement:** As the algorithm progresses, the estimates for the cluster centers ($\mu_k$) and shapes ($\Sigma_k$) shift during each M-step. Consequently, when the next E-step occurs, the conditional probabilities $\gamma_{ik}$ are updated to reflect the new geometry of the mixture components.
* **Soft Responsibility Adjustment:** Rather than assigning a static label, the E-step dynamically recalculates how the total unit probability mass of an observation $x_i$ should be shared among the $K$ clusters. It mathematically recalculates the conditional expectation of cluster occupancy at every iteration, ensuring that the soft cluster memberships continuously adjust to the evolving model.

---
## 8. Parameter Updates (The M-Step)

### Step-by-Step Derivation of GMM Updates

To find the optimal parameter updates, we maximize the expected complete-data log-likelihood function $Q$ derived in the E-step:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$


#### 1. Derivation of $\phi_k^{\text{new}}$
When maximizing $Q$ with respect to the mixing weights $\phi_k$, we must enforce the constraint that they sum to 1 ($\sum_{k=1}^K \phi_k = 1$). We do this using a Lagrange multiplier $\lambda$:

$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k - \lambda \left( \sum_{k=1}^K \phi_k - 1 \right)$$

Taking the partial derivative with respect to $\phi_k$ and setting it to 0:

$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{1}{\lambda} \sum_{i=1}^n \gamma_{ik}$$

Summing both sides over all $k$ to eliminate $\lambda$:

$$\sum_{k=1}^K \phi_k = \frac{1}{\lambda} \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}$$

Since $\sum_{k=1}^K \phi_k = 1$ and $\sum_{k=1}^K \gamma_{ik} = 1$:

$$1 = \frac{1}{\lambda} \sum_{i=1}^n (1) \implies \lambda = n$$

Substituting $\lambda = n$ and defining $N_k = \sum_{i=1}^n \gamma_{ik}$ gives the standard update rule:

$$N_k = \sum_{i=1}^n \gamma_{ik}, \quad \phi_k^{\text{new}} = \frac{N_k}{n}$$


#### 2. Derivation of $\mu_k^{\text{new}}$
Ignoring terms that do not depend on $\mu_k$, the objective function targets the multivariate Gaussian log-density:

$$\log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) = -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) + \text{constant}$$

Taking the vector derivative of $Q$ with respect to $\mu_k$ using the matrix calculus rule $\frac{\partial}{\partial v} (u-v)^T A (u-v) = -2A(u-v)$:

$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \left( \Sigma_k^{-1} (x_i - \mu_k) \right) = 0$$

Multiplying by $\Sigma_k$ and expanding the sum:

$$\sum_{i=1}^n \gamma_{ik} x_i - \sum_{i=1}^n \gamma_{ik} \mu_k = 0 \implies \sum_{i=1}^n \gamma_{ik} x_i = \mu_k \sum_{i=1}^n \gamma_{ik}$$

Using the definition $N_k = \sum_{i=1}^n \gamma_{ik}$ and solving for $\mu_k$ provides:

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$


#### 3. Derivation of $\Sigma_k^{\text{new}}$
Isolating terms depending on $\Sigma_k$ inside the Gaussian log-density:

$$\log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) = -\frac{1}{2} \log |\Sigma_k| -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k)$$

Using the matrix property $v^T A v = \text{tr}(v v^T A)$, we can rewrite the quadratic form:

$$Q_{\Sigma_k} = -\frac{1}{2} \sum_{i=1}^n \gamma_{ik} \left[ \log |\Sigma_k| + \text{tr}\left( (x_i - \mu_k)(x_i - \mu_k)^T \Sigma_k^{-1} \right) \right]$$

Taking the derivative with respect to the precision matrix $\Sigma_k^{-1}$ and setting it to 0:

$$\frac{\partial Q_{\Sigma_k}}{\partial \Sigma_k^{-1}} = \frac{1}{2} \sum_{i=1}^n \gamma_{ik} \left[ \Sigma_k - (x_i - \mu_k)(x_i - \mu_k)^T \right] = 0$$

$$\Sigma_k \sum_{i=1}^n \gamma_{ik} = \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$$

Replacing $\sum_{i=1}^n \gamma_{ik}$ with $N_k$ and substituting the newly updated $\mu_k^{\text{new}}$ yields:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$


### How the Responsibility $\gamma_{ik}$ Acts as a Fractional Membership Weight

The variable $\gamma_{ik}$ behaves exactly like a continuous probability distribution mapping each point to a cluster. When analyzing the update expressions, it acts as a soft statistical weight:

* **In calculating $N_k$:** Instead of counting data points as binary entries ($0$ or $1$), $N_k$ sums the fractional values of $\gamma_{ik}$. It represents the effective total number of samples that realistically belong to cluster $k$.
* **In calculating parameters ($\mu_k, \Sigma_k$):** The update formulas are structurally equivalent to standard weighted sample means and weighted sample covariances. Rather than computing an average over a rigid partition, each observation $x_i$ influences the center and spread of cluster $k$ proportionally to its fractional responsibility score $\gamma_{ik}$. A point near the cluster core plays a massive role ($\gamma_{ik} \approx 1$), while an outlier has almost zero influence ($\gamma_{ik} \approx 0$).

## 9. Interpretation: GMM Clustering as Conditional Updating

Gaussian Mixture Model (GMM) clustering can be viewed as a repeated process of conditional updating through the alternating steps of the EM algorithm:

* **The Mixture Weight as a Prior:** The mixture weight $\phi_k$ is the prior probability of cluster $k$, representing our belief before looking at the features of the data point.
* **Compatibility via Density:** The Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how compatible $x_i$ is with cluster $k$ by evaluating its likelihood under the cluster's current shape and location parameters.
* **The Responsibility as a Posterior:** The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$, computed by updating our prior weight with the component compatibility via Bayes' rule.
* **Soft Assignment Vector:** Stacking these responsibilities yields the soft assignment vector, which is mathematically equivalent to the conditional expectation of the latent configuration vector, $\mathbb{E}[Z_i \mid X_i = x_i]$.
* **Parameter Updates via Weighted Maximization:** The M-step updates the cluster parameters ($\mu_k$, $\Sigma_k$, and $\phi_k$) using these posterior membership probabilities as weights, effectively letting every point fractionally contribute to the definition of every cluster.

### Conclusion
Consequently, we conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.


In [3]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        """
        Initializes the segmenter class with the specified number of clusters.
        """
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=self.n_components, random_state=self.random_state)

    def prepare_data(self, df, feature1='PURCHASES', feature2='CREDIT_LIMIT'):
        """
        Extracts, cleans, scales, and splits the dataset into train and validation sets.
        """
        data_clean = df[[feature1, feature2]].dropna()
        X = data_clean.values

        X_train, X_test = train_test_split(X, test_size=0.20, random_state=self.random_state)

        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)

        print(f"Data split completed. Training shape: {self.X_train_scaled.shape}, Test shape: {self.X_test_scaled.shape}")
        return self.X_train_scaled, self.X_test_scaled

    def fit_em(self):
        """
        Fits the GMM via Expectation-Maximization and reports convergence parameters.
        """
        self.gmm.fit(self.X_train_scaled)
        converged = self.gmm.converged_
        iterations = self.gmm.n_iter_
        print(f"--- EM Execution Results ---")
        print(f"Model Successfully Converged: {converged}")
        print(f"Number of Iterations Required: {iterations}")

    def evaluate_test_set(self):
        """
        Computes the out-of-sample average log-likelihood score.
        """
        test_ll = self.gmm.score(self.X_test_scaled)
        print(f"--- Out-of-Sample Performance ---")
        print(f"Average Log-Likelihood on Test Set: {test_ll:.4f}")
        return test_ll

    def plot_density_heatmap(self):
        """
        1. Generates an empirical 2D Density Heatmap of the training data with marginal distributions.
        """
        df_train = pd.DataFrame(self.X_train_scaled, columns=['PURCHASES (Scaled)', 'CREDIT_LIMIT (Scaled)'])

        fig = px.density_heatmap(
            df_train, x='PURCHASES (Scaled)', y='CREDIT_LIMIT (Scaled)',
            marginal_x='histogram', marginal_y='histogram',
            title='1. Empirical 2D Density Heatmap with Marginal Distributions',
            color_continuous_scale=px.colors.sequential.Viridis
        )
        fig.show()

    def _generate_contour_grid(self, padding=0.5, grid_resolution=200):
        """
        Helper method to evaluate the fine coordinate grid for max posterior responsibility.
        """
        x_min, x_max = self.X_train_scaled[:, 0].min() - padding, self.X_train_scaled[:, 0].max() + padding
        y_min, y_max = self.X_train_scaled[:, 1].min() - padding, self.X_train_scaled[:, 1].max() + padding

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_resolution), np.linspace(y_min, y_max, grid_resolution))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        responsibilities = self.gmm.predict_proba(grid_points)
        max_responsibilities = np.max(responsibilities, axis=1).reshape(xx.shape)

        return xx, yy, max_responsibilities

    def plot_assignment(self, X_data, title_text):
        """
        Helper method to plot data points overlaying the continuous background contour map.
        """
        xx, yy, z_grid = self._generate_contour_grid()
        hard_labels = self.gmm.predict(X_data)

        # Explicitly map discrete integer labels to distinct hex colors to avoid continuous palette errors
        color_palette = ['#E41A1C', '#377EB8', '#4DAF4A']  # Red, Blue, Green
        point_colors = [color_palette[label] for label in hard_labels]

        fig = go.Figure()

        # Add continuous background contour map of maximum responsibilities
        fig.add_trace(go.Contour(
            x=xx[0, :], y=yy[:, 0], z=z_grid,
            colorscale='Cividis', name='Max Responsibility',
            colorbar=dict(title='Max gamma_ik'),
            opacity=0.7, contours_showlines=False
        ))

        # Scatter overlay of data points color-coded cleanly
        fig.add_trace(go.Scatter(
            x=X_data[:, 0], y=X_data[:, 1],
            mode='markers',
            marker=dict(
                size=5,
                color=point_colors,
                line=dict(width=0.5, color='black')
            ),
            name='Observations'
        ))

        fig.update_layout(
            title=title_text,
            xaxis_title='PURCHASES (Scaled)',
            yaxis_title='CREDIT_LIMIT (Scaled)',
            template='plotly_white'
        )
        fig.show()

# --- Simulation Execution Block ---
np.random.seed(42)
dummy_df = pd.DataFrame({
    'PURCHASES': np.concatenate([np.random.normal(500, 200, 500), np.random.normal(2500, 600, 300), np.random.normal(100, 50, 200)]),
    'CREDIT_LIMIT': np.concatenate([np.random.normal(2000, 500, 500), np.random.normal(7000, 1500, 300), np.random.normal(12000, 2000, 200)])
})

segmenter = GMMFinancialSegmenter(n_components=3)
X_train, X_test = segmenter.prepare_data(dummy_df)
segmenter.fit_em()
segmenter.evaluate_test_set()

# Generate the interactive figures safely
segmenter.plot_density_heatmap()
segmenter.plot_assignment(X_train, "2. Training Assignment Plot Over Posterior Contours")
segmenter.plot_assignment(X_test, "3. Test Assignment Plot Over Posterior Contours")

Data split completed. Training shape: (800, 2), Test shape: (200, 2)
--- EM Execution Results ---
Model Successfully Converged: True
Number of Iterations Required: 3
--- Out-of-Sample Performance ---
Average Log-Likelihood on Test Set: -0.7881
